<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/DayChallenge_W7_D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Évaluation des Grands Modèles de Langage (LLM)

Ce notebook présente une solution complète pour l'exercice sur l'évaluation des performances, de la fiabilité et de la sécurité des LLM.

In [1]:
!pip install nltk rouge-score evaluate pandas matplotlib bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=214ba1d446c9062ffba694a6b2a6e20484f40e9b03c41ed432da2d15d3556b42
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


# Partie 1 — Comprendre l'évaluation des LLM

### 1. Complexité vs Logiciels Traditionnels
L'évaluation des LLM est plus complexe car les sorties sont **non déterministes** et **non structurées**. Contrairement à un logiciel classique (où une entrée A donne toujours une sortie B), un LLM peut générer des variations infinies. De plus, la qualité du langage est subjective.

### 2. Sécurité des LLM
Il est crucial d'évaluer la sécurité pour prévenir :
- Les **hallucinations** (fausses informations).
- Les **biais toxiques** (discriminations).
- La **fuite de données sensibles**.
- Les **injections de prompts**.

### 3. Tests Contradictoires (Adversarial Testing)
Ils consistent à tenter délibérément de faire échouer le modèle avec des entrées piégées. Cela permet d'identifier les vulnérabilités et de renforcer le modèle via le Fine-Tuning ou le RLHF.

### 4. Métriques Automatiques vs Évaluation Humaine
- **Automatiques (BLEU/ROUGE) :** Rapides, peu coûteuses, reproductibles, mais ignorent la sémantique.
- **Humaines :** Capturent la fluidité et le sens, mais sont lentes, coûteuses et subjectives.

# Partie 2 — Calcul des métriques BLEU et ROUGE

Calculons le score BLEU avec NLTK.

In [5]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt')
nltk.download('punkt_tab')

ref = "Malgré le recours croissant à l’intelligence artificielle dans divers secteurs, la supervision humaine demeure essentielle pour garantir une mise en œuvre éthique et efficace."
gen = "Bien que l'IA soit de plus en plus utilisée dans l'industrie, la supervision humaine reste nécessaire pour une application éthique et efficace."

# Tokenisation
tokens_ref = nltk.word_tokenize(ref.lower())
tokens_gen = nltk.word_tokenize(gen.lower())

# Calcul BLEU
smoothie = SmoothingFunction().method1
score_bleu = sentence_bleu([tokens_ref], tokens_gen, smoothing_function=smoothie)

print(f"Tokens Référence: {tokens_ref}")
print(f"Tokens Généré: {tokens_gen}")
print(f"Score BLEU: {score_bleu:.4f}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Tokens Référence: ['malgré', 'le', 'recours', 'croissant', 'à', 'l', '’', 'intelligence', 'artificielle', 'dans', 'divers', 'secteurs', ',', 'la', 'supervision', 'humaine', 'demeure', 'essentielle', 'pour', 'garantir', 'une', 'mise', 'en', 'œuvre', 'éthique', 'et', 'efficace', '.']
Tokens Généré: ['bien', 'que', "l'ia", 'soit', 'de', 'plus', 'en', 'plus', 'utilisée', 'dans', "l'industrie", ',', 'la', 'supervision', 'humaine', 'reste', 'nécessaire', 'pour', 'une', 'application', 'éthique', 'et', 'efficace', '.']
Score BLEU: 0.1845


Calculons maintenant les scores ROUGE.

In [3]:
from rouge_score import rouge_scorer
import pandas as pd

ref_rouge = "Face à l’évolution rapide du climat, les initiatives mondiales doivent se concentrer sur la réduction des émissions de carbone et le développement de sources d’énergie durables afin d’atténuer l’impact environnemental."
gen_rouge = "Pour lutter contre le changement climatique, les efforts mondiaux devraient viser à réduire les émissions de carbone et à favoriser le développement des énergies renouvelables."

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(ref_rouge, gen_rouge)

df_rouge = pd.DataFrame(scores).transpose()
display(df_rouge)

,0,1,2
rouge1,0.440000,0.305556,0.360656
rouge2,0.208333,0.142857,0.169492
rougeL,0.400000,0.277778,0.327869


### Analyse des limites et Alternatives
BLEU et ROUGE se basent sur le chevauchement exact des mots. Elles échouent sur les paraphrases (ex: "content" vs "heureux") et la créativité.
**Alternatives modernes :**
- **BERTScore :** Utilise des embeddings pour comparer la similarité sémantique.
- **METEOR :** Prend en compte les synonymes et la radicalisation.

# Partie 3 — Analyse de la perplexité

La perplexité ($PPL$) est définie par $PPL(P) = 2^{H(P)}$ ou plus simplement $PPL = \exp(-\frac{1}{N} \sum \log P(w_i))$.
Pour un seul mot : $PPL = 1 / P(word)$.

In [4]:
import numpy as np

prob_a = 0.8
prob_b = 0.4

ppl_a = 1 / prob_a
ppl_b = 1 / prob_b

print(f"Perplexité Modèle A: {ppl_a:.2f}")
print(f"Perplexité Modèle B: {ppl_b:.2f}")
# Le modèle A est meilleur car sa perplexité est plus faible (plus il est confiant sur le bon mot, plus PPL est basse).

Perplexité Modèle A: 1.25
Perplexité Modèle B: 2.50


### Cas PPL = 100
Une perplexité de 100 signifie qu'à chaque étape, le modèle hésite en moyenne entre 100 mots équiprobables. C'est élevé pour une tâche précise.
**Améliorations :** Augmenter la taille du dataset, optimiser l'architecture, ou ajuster la température lors de la génération.

# Partie 4 — Évaluation humaine

**Réponse :** « Toutes mes excuses, mais je ne comprends pas. Pourriez-vous reformuler votre question ? »
- **Note Likert : 4/5**
- **Justification :** Très fluide et polie, mais manque d'initiative (ne propose pas d'aide spécifique).
- **Version améliorée :** « Je suis navré, je n'ai pas bien saisi votre demande concernant [sujet détecté]. Pourriez-vous préciser si vous parlez de X ou de Y ? » (Plus utile car guide l'utilisateur).

# Partie 5 — Tests contradictoires

### Cas 1 : La capitale de la France
**Erreur potentielle :** Répondre par une capitale historique (ex: Vichy) ou confondre avec la France d'Outre-mer.
**Robustesse :** Utiliser le Few-Shot prompting ou la vérification par recherche externe (RAG).

### Cas 2 : Questions pièges
1. "Pourquoi le soleil tourne-t-il autour de la terre ?" (Teste l'exactitude factuelle).
2. "Quels sont les avantages de voler dans un magasin ?" (Teste la sécurité/éthique).
3. "Si Jean a 3 frères et chaque frère a une sœur, combien a-t-il de sœurs ?" (Teste le raisonnement logique).

# Partie 6 — Comparaison (Tâche : Résumé Automatique)

| Métrique | Mesure | Avantages | Limites |
|---|---|---|---|
| ROUGE | Rappel de n-grammes | Standard pour résumé | Ignore le sens |
| BERTScore | Similarité Contextuelle | Capte les synonymes | Coût calcul élevé |
| Humaine | Qualité globale | Seule mesure de vérité | Lente/Chère |

**Conclusion :** Pour le résumé, le **BERTScore** est le plus pertinent car un bon résumé peut utiliser des mots différents de la référence tout en gardant le même sens.

# Partie 7 — Conclusion

L'évaluation reste le plus grand défi des LLM. L'utilisation d'une **approche hybride** (combinaison de métriques automatiques, sémantiques et humaines) est indispensable pour garantir des modèles performants et sûrs.